# Netflix Content Analysis — Professional EDA Notebook

**Goal.** This notebook is a complete, interview-grade exploratory data analysis (EDA) of the
[Netflix Titles dataset](https://www.kaggle.com/datasets/shivamb/netflix-shows) (Shivam Bansal / Kaggle).
Every row is one movie or TV show available on Netflix in late 2021.

Alongside the analysis, **every code cell** is explained in three parts:

- **What this cell does** — the intent of the code.
- **Pandas functions used** — the specific pandas methods, and why each is the right tool.
- **How to answer this in an interview** — a short, real-world answer you can reuse.

**Dataset columns**

| Column | Meaning |
|---|---|
| `show_id` | Unique identifier |
| `type` | `Movie` or `TV Show` |
| `title` | Title name |
| `director` | Director(s) |
| `cast` | Main cast |
| `country` | Producing country / countries |
| `date_added` | When it was added to Netflix |
| `release_year` | Original release year |
| `rating` | Age rating (e.g. `PG-13`) |
| `duration` | Minutes (movies) or seasons (shows) |
| `listed_in` | Genre / category |
| `description` | Short plot summary |

**Business questions we answer**

1. What is the Movies vs TV Shows mix in the catalog?
2. How has the catalog grown over time?
3. Which countries produce the most content?
4. Which genres dominate?
5. How is content rated?
6. What are typical movie runtimes and TV-show lengths?

**How to run**

1. Download `netflix_titles.csv` from the Kaggle link above and place it in `data/` at the project root.
2. This notebook looks for that file first; if it is missing it transparently generates a small,
   clearly-labelled **synthetic sample** so every cell still runs end-to-end.
3. Charts are saved automatically to `outputs/figures/`.

## Table of contents

1. [Project Introduction](#1-project-introduction)
2. [Import Libraries](#2-import-libraries)
3. [Load Data](#3-load-data)
4. [Initial Exploration](#4-initial-exploration)
5. [Data Cleaning](#5-data-cleaning)
6. [Exploratory Data Analysis](#6-exploratory-data-analysis)
7. [Key Insights](#7-key-insights)
8. [Conclusion](#8-conclusion)

## 2. Import Libraries

**What this cell does**

Imports every library used in this notebook: `pandas` for tabular data, `numpy` for numerical
arrays and random-number generation, `matplotlib` + `seaborn` for plotting, and `pathlib` for
filesystem paths. It then applies one consistent chart theme and figure size so that all plots
share a professional, uniform look.

**Pandas functions used**

Importing does not call any pandas functions yet, but it does two important things:

- `import pandas as pd` binds the whole library to the alias `pd`. Every later call
  (`pd.read_csv`, `pd.to_datetime`, ...) goes through this alias.
- `import matplotlib.pyplot as plt` and `import seaborn as sns` create the plotting namespaces.
  Seaborn sits on top of matplotlib and adds statistical plots plus better visual defaults.

**How to answer this in an interview**

If asked 'Which libraries would you use for EDA and why?', answer: pandas for tabular
manipulation, numpy for numerical arrays, and seaborn on top of matplotlib for charts because it
provides statistical plotting out of the box (`sns.histplot`, `sns.barplot`, ...) with cleaner
defaults. Mentioning `plt.rcParams` and `sns.set_theme` shows you know how to standardise style.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 3. Load Data

**What this cell does**

Defines the dataset path (`data/netflix_titles.csv`) and the chart output folder. Then it defines
a reusable `load_netflix()` helper that reads the real CSV or — if the file is absent — builds a
**reproducible synthetic fallback** (fixed random seed) so the notebook always runs. Finally it
loads the data and prints the number of rows and columns.

**Pandas functions used**

- `pd.read_csv(path)` — the standard way to import a CSV into a `DataFrame` (a two-dimensional,
  labelled table). All 12 Netflix columns arrive as columns of this object.
- `DataFrame.shape` — returns `(n_rows, n_columns)` so we can confirm the file was read correctly.
- `DataFrame.head()` — prints the first 5 rows so we can visually sanity-check the schema.
- `np.random.default_rng(seed)` — not pandas, but numpy's reproducible random generator, used only
  to build the demo fallback when the real file is absent.

**How to answer this in an interview**

When asked 'How do you load data in pandas?', say: `pd.read_csv` for CSVs, `pd.read_excel` for
Excel, `pd.read_sql` for databases, and `pd.read_json` for JSON. Always pair the load with
`df.shape` and `df.head()` to **verify** rows, columns, and dtypes look right before analysing.
Interviewers reward the habit of checking data immediately after loading.

In [ ]:
from datetime import datetime, timedelta

BASE = Path.cwd().parent                     # project root: notebooks/ -> project/
DATA_PATH = BASE / 'data' / 'netflix_titles.csv'
FIG_DIR = BASE / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_netflix(csv_path: Path) -> pd.DataFrame:
    '''Read the real CSV, or build a clearly-labelled synthetic sample.'''
    if csv_path.exists():
        print(f'Loaded real dataset from {csv_path}')
        return pd.read_csv(csv_path)

    print(f'[FALLBACK] {csv_path.name} not found - generating synthetic demo data.')
    print('Place netflix_titles.csv in data/ to use the full dataset.')
    rng = np.random.default_rng(42)
    n = 6_200

    types = rng.choice(['Movie', 'TV Show'], size=n, p=[0.70, 0.30])
    ratings = rng.choice(
        ['TV-MA', 'TV-14', 'TV-PG', 'R', 'PG-13', 'PG', 'G'],
        size=n, p=[0.35, 0.25, 0.10, 0.10, 0.10, 0.07, 0.03],
    )
    countries = rng.choice(
        ['United States', 'India', 'United Kingdom', 'Canada', 'Japan', 'France',
         'South Korea', 'Spain', 'Germany', 'Australia'],
        size=n, p=[0.38, 0.15, 0.08, 0.06, 0.06, 0.05, 0.05, 0.05, 0.04, 0.03],
    )
    start = datetime(2008, 1, 1)
    date_added = [
        (start + timedelta(days=int(rng.integers(0, 14 * 365)))).strftime('%B %d, %Y')
        for _ in range(n)
    ]

    duration = []
    for t in types:
        if t == 'Movie':
            duration.append(f'{int(rng.integers(45, 230))} min')
        else:
            s = int(rng.integers(1, 10))
            duration.append(f'{s} Season' + ('' if s == 1 else 's'))

    genres = ['Dramas', 'Comedies', 'Action & Adventure', 'Thrillers',
              'Documentaries', 'Reality TV', 'Children & Family Movies',
              'Sci-Fi & Fantasy', 'Romantic Movies', 'Crime TV Shows']
    listed_in = []
    for _ in range(n):
        k = int(rng.integers(1, 4))
        listed_in.append(', '.join(rng.choice(genres, size=k, replace=False)))

    return pd.DataFrame({
        'show_id': [f's{i}' for i in range(1, n + 1)],
        'type': types,
        'title': [f'Demo Title {i}' for i in range(1, n + 1)],
        'director': rng.choice(['A. Director', 'B. Filmmaker', 'C. Auteur'], size=n),
        'cast': rng.choice(['Actor One', 'Actor Two', 'Actor Three'], size=n),
        'country': countries,
        'date_added': date_added,
        'release_year': rng.integers(1940, 2022, size=n),
        'rating': ratings,
        'duration': duration,
        'listed_in': listed_in,
        'description': rng.choice(['A gripping tale.', 'A comedy of errors.'], size=n),
    })


df_raw = load_netflix(DATA_PATH)
print(f'Dataset size: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head()

## 4. Initial Exploration

**What this cell does**

Checks the dimensions of the raw table with `df_raw.shape` before any cleaning, so we know how
many titles (rows) we are dealing with and how many attributes (columns) each has.

**Pandas functions used**

- `DataFrame.shape` — a tuple `(rows, columns)`. This is the fastest 'did my load work?' check.

**How to answer this in an interview**

For 'How do you sanity-check a dataset you just loaded?', answer: `df.shape` for size,
`df.head()` for content, `df.info()` for dtypes and nulls, then `df.describe()` for summary
statistics. Always do the load + check in that order.

In [ ]:
df_raw.shape

**What this cell does**

Displays the first 10 and last 5 rows. `head()` shows how the very first rows are structured;
`tail()` verifies the file's end was read correctly and that boundary rows are not malformed.

**Pandas functions used**

- `DataFrame.head(n)` — first `n` rows (default 5).
- `DataFrame.tail(n)` — last `n` rows.

**How to answer this in an interview**

If asked 'How do you get a feel for a new dataset?', mention that you look at `head()` and `tail()`
for edges, `.sample()` for a random look, then move immediately to `info()` for the schema and
`nunique()` for the cardinality of categorical columns.

In [ ]:
df_raw.head(10)
df_raw.tail(5)

**What this cell does**

Prints the internal structure of the DataFrame: column names, number of non-null values per
column, and each column's data type. Non-null counts immediately reveal which columns contain
missing values before we even run a dedicated check.

**Pandas functions used**

- `DataFrame.info()` — a summary of axis labels, dtypes, non-null counts, and total memory usage.
  One call replaces several manual checks.

**How to answer this in an interview**

'How do you inspect a DataFrame's schema?' — say `df.info()` and read it out loud: object columns
are likely categorical, `int64`/`float64` are numeric, and `datetime64` is time-based. Explain
that counting non-nulls here tells you where cleaning will be needed (e.g. `director`, `cast`,
`country` are commonly incomplete in this dataset).

In [ ]:
df_raw.info()

**What this cell does**

`df.describe(include='all')` computes summary statistics for **every** column, including text
columns (`unique`, `top`, `freq`) which the default numeric-only output omits. `df.nunique()`
counts distinct values per column, which is how we spot high-cardinality vs low-cardinality
fields (e.g. `title` is nearly unique per row, while `type` has only 2 values).

**Pandas functions used**

- `DataFrame.describe(include='all')` — summary stats: `count`, `mean`, `std`, `min`, quartiles,
  `max` for numeric data, plus `unique`, `top`, `freq` for object data.
- `DataFrame.nunique()` — counts unique values per column (excludes NaN by default).

**How to answer this in an interview**

If asked 'Which columns are categorical in the Netflix dataset?', say: `type`, `rating`,
`country`, `director`, `cast`, `listed_in` are categorical; `duration` is semi-numeric (mixed
units) and `date_added` is a datetime stored as text. Support the answer with `df.nunique()`;
`type` has 2 unique values, `rating` about a dozen, `title` nearly the row-count.

In [ ]:
df_raw.describe(include='all')
df_raw.nunique()

**What this cell does**

Counts nulls per column with `isnull().sum()`, converts those counts to percentages, and stacks
both into a small DataFrame sorted from most- to least-missing. This gives a single table that
tells us *where* cleaning is needed and *how bad* the problem is.

**Pandas functions used**

- `DataFrame.isnull()` — an element-wise NaN mask (True where a value is missing).
- `Series.sum()` — sums the True values, so `df.isnull().sum()` yields missing counts per column.
- `len(df_raw)` — the row count, used to compute the missing percentage.
- `pd.DataFrame({...})` — builds a new table from two Series (count and percent).
- Boolean filter `missing_summary['missing'] > 0` — keeps only columns that actually have nulls.
- `Series.sort_values(ascending=False)` — puts the worst offenders on top.

**How to answer this in an interview**

For 'How do you handle missing values?', first diagnose with `df.isnull().sum()` and compute the
percentage; only then choose a strategy — drop rows when the share is tiny, fill categoricals with
`'Unknown'`, or leave dates as NaT. Never fill blindly: understand *why* the data is missing.

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_summary = pd.DataFrame({'missing': missing, 'percent': missing_pct})
missing_summary = missing_summary[missing_summary['missing'] > 0].sort_values('missing', ascending=False)
missing_summary

**What this cell does**

Counts fully-duplicated rows, then counts rows duplicated on the business key `(title, type)`.
The second check catches near-duplicates: the same title listed twice with tiny metadata
differences, which a whole-row check would miss.

**Pandas functions used**

- `DataFrame.duplicated()` — returns a boolean Series marking duplicates (True = a row seen
  earlier in the frame).
- `Series.sum()` — counts the True values.
- `DataFrame.duplicated(subset=['title', 'type'])` — restricts the duplicate check to the
  specified columns, the real-world business key for a catalog like Netflix.

**How to answer this in an interview**

'How do you detect duplicates?' — answer: `df.duplicated().sum()` for exact duplicates, then
`df.duplicated(subset=[...])` on the columns that define a unique record. For Netflix, `title`
plus `type` is the natural business key; the catalog should never hold the same movie twice.

In [ ]:
df_raw.duplicated().sum()
df_raw.duplicated(subset=['title', 'type']).sum()

## 5. Data Cleaning

**What this cell does**

Creates a working copy of the raw frame. Cleaning should never mutate the original dataset:
keeping `df_raw` untouched means we can always re-run a cleaning step or audit it later.

**Pandas functions used**

- `DataFrame.copy()` — returns an independent copy. Without it, some assignments on a sliced
  frame raise `SettingWithCopyWarning` and can silently fail to write.

**How to answer this in an interview**

'Why do you call `.copy()`?' — say you always preserve raw data for reproducibility and auditing,
and that operations on sliced frames without a copy can trigger `SettingWithCopyWarning`. It is
cheap insurance in any pipeline.

In [ ]:
df = df_raw.copy()
print('Working copy created from raw data.')
df.head(2)

**What this cell does**

Fills missing values in the categorical columns (`director`, `cast`, `country`, `rating`,
`listed_in`) with the sentinel string `'Unknown'`, and empty descriptions with `''`. This keeps
every title in the analysis (zero rows dropped) while making missingness explicit.

**Pandas functions used**

- `DataFrame.fillna(value)` — replaces NaN with a given value. Here `'Unknown'` is deliberately a
  *labelled sentinel*, not a guess.
- A simple `for col in categorical_cols:` loop applies the same policy to many columns at once.

**How to answer this in an interview**

'Fill or drop missing values?' — the interviewer wants the reasoning: fill categoricals with
`'Unknown'` when dropping would lose too many rows and the label means 'not known' rather than
'not applicable'; drop rows only when the null share is tiny; for numeric columns consider the
median. Explain the *why*, not just the code.

In [ ]:
categorical_cols = ['director', 'cast', 'country', 'rating', 'listed_in']
for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')

df['description'] = df['description'].fillna('')

remaining = df.isnull().sum()
remaining[remaining > 0]

**What this cell does**

Converts the free-text `date_added` (e.g. `September 15, 2021`) into a real `datetime64` column,
then extracts `year_added` and `month_added`. Converting text to datetimes is what unlocks
time-series analysis such as 'titles added per year'.

**Pandas functions used**

- `pd.to_datetime(series, format='%B %d, %Y', errors='coerce')` — parses strings into datetimes
  using an explicit format; `errors='coerce'` turns unparseable values into `NaT` instead of
  crashing the whole operation.
- `.dt` accessor — reaches inside the datetime column: `.dt.year`, `.dt.month`.
- `Series.astype('Int64')` — the **nullable** integer dtype, which can hold NaN after conversion.

**How to answer this in an interview**

'Why convert strings to datetime?' — because you cannot resample, sort chronologically, or do date
arithmetic on strings. Mention `errors='coerce'` to show you anticipate dirty data, and the
nullable `Int64` dtype to show you understand how pandas stores missing integers.

In [ ]:
df['date_added'] = pd.to_datetime(df['date_added'], format='%B %d, %Y', errors='coerce')
df['year_added'] = df['date_added'].dt.year.astype('Int64')
df['month_added'] = df['date_added'].dt.month

df[['date_added', 'year_added', 'month_added']].head()

**What this cell does**

Splits the mixed-unit `duration` column into two clean numeric fields: `movie_duration_minutes`
(films) and `tv_seasons` (shows). Movies store runtimes like `90 min`; shows store `2 Seasons`.
Mixing the two units in one column rules out any numeric aggregation, so we separate them.

**Pandas functions used**

- Boolean mask `df['type'] == 'Movie'` — selects only movie rows for the operation.
- `DataFrame.loc[mask, col]` — a row-and-column indexer; here it targets only the movie rows.
- `Series.str.replace(' min', '', regex=False)` — the string accessor; `regex=False` treats the
  pattern literally, avoiding regex escaping.
- `Series.astype('Float64')` — nullable float, so TV-show rows stay NaN without losing nullability.

**How to answer this in an interview**

'How do you handle messy text that encodes numbers?' — normalize units first (`str.replace` on
literal tokens), then coerce with `astype`. Emphasize that you keep one numeric column per unit
and use masks (`loc[mask, col]`) so you never accidentally mix minutes with seasons.

In [ ]:
movie_mask = df['type'] == 'Movie'
df.loc[movie_mask, 'movie_duration_minutes'] = (
    df.loc[movie_mask, 'duration']
    .str.replace(' min', '', regex=False)
    .astype('Float64')
)

show_mask = df['type'] == 'TV Show'
df.loc[show_mask, 'tv_seasons'] = (
    df.loc[show_mask, 'duration']
    .str.replace(' Season', '', regex=False)
    .str.replace('s', '', regex=False)
    .astype('Float64')
)

df[['type', 'duration', 'movie_duration_minutes', 'tv_seasons']].sample(8, random_state=1)

**What this cell does**

Bins `release_year` into decades (`1950s`, `1960s`, ..., `2020s`) using `pd.cut`. Decades smooth
out yearly noise and let us study long-term catalog composition instead of single-year quirks.

**Pandas functions used**

- `pd.cut(series, bins, labels, right=False)` — buckets a continuous variable into labelled
  intervals. `right=False` makes intervals `[start, end)`, so 2020 belongs to the `2020s`.
- `Series.value_counts()` — then counts how many titles fall in each decade.

**How to answer this in an interview**

'How do you bin a numeric column?' — answer `pd.cut` for value-based bins (with explicit labels so
plots read nicely) and `pd.qcut` when you want equal-frequency bins (e.g. quartiles). Mention
`right=False` to control boundary behaviour — that detail separates you from juniors.

In [ ]:
bins = list(range(1950, 2040, 10))
labels = [f'{b}s' for b in bins[:-1]]

df['release_decade'] = pd.cut(
    df['release_year'],
    bins=bins,
    labels=labels,
    right=False,
).astype('object')

df['release_decade'].value_counts()

**What this cell does**

Explodes the multi-value columns `country` and `listed_in`: every comma-separated value gets its
own row. A title listed under `Dramas, Comedies` therefore contributes one row to each genre
bucket, which enables clean `value_counts()` answers for 'top countries' and 'top genres'.

**Pandas functions used**

- `DataFrame.assign(col=expr)` — adds a new column (here, the split list) to a copy.
- `Series.str.split(', ')` — splits strings into Python lists.
- `DataFrame.explode(col)` — one row per list element, repeating the other columns.
- `len(df)` — lets us print how much the frame grew.

**How to answer this in an interview**

'An attribute column contains several values in one cell — how do you analyse it?' — say: split
with `str.split`, then `explode` to one row per value, then `value_counts`. Also flag that the
exploded frame is a derived, longer table used only for those counts — the master table still has
one row per title.

In [ ]:
df_countries = df.assign(countries=df['country'].str.split(', ')).explode('countries')
df_genres = df.assign(genres=df['listed_in'].str.split(', ')).explode('genres')

print(f'rows: master={len(df):,} | countries={len(df_countries):,} | genres={len(df_genres):,}')
df_genres.head()

**What this cell does**

Final quality gate on the cleaned frame: shape, remaining nulls, duplicates, and dtypes. This is
the 'clean, verify, then analyse' step — no chart or model should be trusted on unvalidated data.

**Pandas functions used**

- `DataFrame.isnull().sum()` — confirms which columns still hold missing values (only `date_added`
  and the derived numeric duration columns, which is intended).
- `DataFrame.duplicated().sum()` — confirms no duplicate rows were introduced.
- `DataFrame.dtypes` — prints the final dtype for every column.

**How to answer this in an interview**

'How do you know cleaning succeeded?' — compare shape before/after, confirm remaining nulls are
only the *intended* ones, confirm zero duplicates, and re-check dtypes (`datetime64` for dates,
`Int64`/`Float64` for numerics). Verifying before any analysis is a core habit.

In [ ]:
print('Shape after cleaning:', df.shape)
print('
Remaining missing values (intended):')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('
Duplicate rows:', df.duplicated().sum())
print('
Data types:')
print(df.dtypes)

## 6. Exploratory Data Analysis

**What this cell does**

Answers business question #1: what fraction of the catalog is movies vs TV shows. It prints
counts and percentage shares, then draws a bar chart in Netflix red and labels each bar with the
value.

**Pandas functions used**

- `Series.value_counts()` — counts occurrences by category, sorted most-to-least.
- `Series.value_counts(normalize=True)` — returns relative frequencies (0..1).
- `Series.mul(100).round(1)` — turns the fractions into rounded percentages.
- `sns.countplot(data, x=...)` — seaborn's categorical bar-counting plot (built on matplotlib).

**How to answer this in an interview**

'What's your first cut of a categorical column?' — `value_counts()` plus `normalize=True` for the
share. Then say: for Netflix, movies are roughly 70% of the catalog — a classic headline insight
the interviewer will expect you to find quickly.

In [ ]:
type_counts = df['type'].value_counts()
type_share = df['type'].value_counts(normalize=True).mul(100).round(1)

print('Counts:'); print(type_counts)
print('
Share (%):'); print(type_share)

fig, ax = plt.subplots()
sns.countplot(data=df, x='type', color='#E50914', ax=ax)
ax.set_title('Netflix catalog: Movies vs TV Shows', pad=12)
ax.set_xlabel('Content type')
ax.set_ylabel('Number of titles')
for i, v in enumerate(type_counts.values):
    ax.text(i, v + 25, f'{v:,}', ha='center', fontweight='bold')
plt.savefig(FIG_DIR / 'content_type_split.png', bbox_inches='tight')
plt.show()

**What this cell does**

Answers business question #2: how many titles were added per year, split by type. It groups the
cleaned frame by `(year_added, type)`, counts rows per group, and draws two lines so we can compare
the growth trajectories of movies and shows over time.

**Pandas functions used**

- `DataFrame.groupby(['year_added', 'type'])` — splits rows into groups by those two columns.
- `GroupBy.size()` — count of rows per group.
- `DataFrame.reset_index(name='count')` — turns the grouped index back into columns and names the
  count column.
- `DataFrame.dropna(subset=['year_added'])` — removes titles whose add date could not be parsed,
  so gaps don't look like zeros.
- `sns.lineplot(data, x, y, hue=...)` — line chart with one series per category.
- `mticker.MultipleLocator(2)` — spaces the x-axis ticks every 2 years for readability.

**How to answer this in an interview**

'How do you summarise counts by two categorical variables?' — `groupby([...]).size()` returns a
Series; add `.reset_index(name='count')` to get a tidy table. Also mention `pivot_table` and
`value_counts(normalize=True)` as alternatives, and emphasize dropping NaN before plotting.

In [ ]:
yearly = (
    df.groupby(['year_added', 'type'])
    .size()
    .reset_index(name='count')
    .dropna(subset=['year_added'])
)

print(yearly.head(10))

fig, ax = plt.subplots()
sns.lineplot(data=yearly, x='year_added', y='count', hue='type',
             marker='o', linewidth=2.5, ax=ax)
ax.set_title('Titles added to Netflix per year', pad=12)
ax.set_xlabel('Year added')
ax.set_ylabel('Number of titles added')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.savefig(FIG_DIR / 'yearly_growth.png', bbox_inches='tight')
plt.show()

**What this cell does**

Answers business question #3: the top 10 producing countries, computed on the exploded
`df_countries` table so multi-country titles count for each country. Rendered as a horizontal bar
chart for readable labels.

**Pandas functions used**

- `Series.value_counts().head(10)` — category frequencies, sliced to the top 10.
- `sns.barplot(x=values, y=labels)` — horizontal bars; labels read naturally left-to-right.
- `plt.savefig(...)` — persists the chart to `outputs/figures/` for reports or a portfolio.

**How to answer this in an interview**

For 'Which country dominates Netflix production?' the expected answer is the United States,
followed by India. Mention you *exploded* `country` first because a title like 'United States,
India' is legitimately counted for both. In the real dataset a share of titles are `'Unknown'`;
say you would filter that sentinel out of the displayed top-10.

In [ ]:
country_counts = df_countries['countries'].value_counts().head(10)
print(country_counts)

fig, ax = plt.subplots()
sns.barplot(x=country_counts.values, y=country_counts.index, color='#E50914', ax=ax)
ax.set_title('Top 10 content-producing countries', pad=12)
ax.set_xlabel('Number of titles')
ax.set_ylabel('Country')
plt.savefig(FIG_DIR / 'top_countries.png', bbox_inches='tight')
plt.show()

**What this cell does**

Answers business question #4: the 10 most frequent genres, computed on the exploded `df_genres`
table so a title in multiple genres contributes to each of them.

**Pandas functions used**

- `Series.value_counts().head(10)` — top categories by frequency.
- `sns.barplot` horizontal — comfortable reading for many category labels.
- `plt.savefig(..., bbox_inches='tight')` — crops tightly so labels are never cut off.

**How to answer this in an interview**

'What are Netflix's most common genres?' — Dramas and Comedies dominate. Add the caveat that
`listed_in` is multi-valued, so counts across genres sum to more than the number of titles — a
distinction interviewers use to test whether you understood the explosion step.

In [ ]:
genre_counts = df_genres['genres'].value_counts().head(10)
print(genre_counts)

fig, ax = plt.subplots()
sns.barplot(x=genre_counts.values, y=genre_counts.index, color='#E50914', ax=ax)
ax.set_title('Top 10 genres on Netflix', pad=12)
ax.set_xlabel('Number of titles')
ax.set_ylabel('Genre')
plt.savefig(FIG_DIR / 'top_genres.png', bbox_inches='tight')
plt.show()

**What this cell does**

Answers business question #5: the distribution of age ratings. Counting ratings reveals how
family-friendly (or adult-skewed) the catalog is — a question product teams genuinely care about.

**Pandas functions used**

- `Series.value_counts()` — counts per rating.
- Horizontal `sns.barplot` — long rating labels (`TV-PG`, `PG-13`, ...) fit best on the y-axis.

**How to answer this in an interview**

The expected story: TV-MA and TV-14 are the two most common ratings, so the catalog skews
adult/teen rather than child-oriented. Then volunteer the honest caveat that some rows had
`'Unknown'` ratings and how that could slightly bias the mix — interviewers love a self-awareness
caveat.

In [ ]:
rating_counts = df['rating'].value_counts()
print(rating_counts)

fig, ax = plt.subplots()
sns.barplot(x=rating_counts.values, y=rating_counts.index, color='#E50914', ax=ax)
ax.set_title('Content rating distribution', pad=12)
ax.set_xlabel('Number of titles')
ax.set_ylabel('Rating')
plt.savefig(FIG_DIR / 'content_ratings.png', bbox_inches='tight')
plt.show()

**What this cell does**

Answers business question #6 for movies: describes movie runtimes and plots their distribution. A
`describe()` gives the five-number summary; the histogram + KDE shows the typical shape (roughly
bell-shaped around 90-120 minutes for Netflix).

**Pandas functions used**

- `Series.dropna()` — drops TV shows' missing duration values before computing statistics.
- `Series.describe().round(1)` — mean, std, min, quartiles, max for the numeric column.
- `Series.median()` — a robust centre measure, drawn as a dashed reference line.
- `sns.histplot(x, bins=30, kde=True)` — histogram with a kernel-density estimate, the standard
  professional look in current seaborn.

**How to answer this in an interview**

'What is the typical Netflix movie length?' — roughly 90-120 minutes, median around 98. Explain
that you report the *median* rather than the mean because runtimes are right-skewed by very long
films, and back it up with the histogram shape. That statistics talk is what separates stronger
candidates.

In [ ]:
movie_dur = df['movie_duration_minutes'].dropna()
print(movie_dur.describe().round(1))

fig, ax = plt.subplots()
sns.histplot(movie_dur, bins=30, kde=True, color='#E50914', ax=ax)
ax.axvline(movie_dur.median(), color='#564D4D', linestyle='--',
           label=f'Median: {movie_dur.median():.0f} min')
ax.set_title('Distribution of movie durations', pad=12)
ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Number of movies')
ax.legend()
plt.savefig(FIG_DIR / 'movie_duration.png', bbox_inches='tight')
plt.show()

**What this cell does**

Answers business question #6 for shows: how many titles have 1, 2, 3, ... seasons. Most shows have
a single season, and the frequency drops quickly — a signature of Netflix's binge-friendly,
mostly single-season catalog.

**Pandas functions used**

- `Series.dropna().astype('int')` — cleans the nullable float season counts into plain integers.
- `Series.value_counts().sort_index()` — counts per season number, ordered from 1 to N.
- `sns.countplot(x=...)` — a categorical bar chart of the season-count frequency.

**How to answer this in an interview**

The takeaway: the modal show has 1 season, and 2+ season shows become increasingly rare. This same
logic transfers to any 'how long does the typical unit last' question (churn cohorts, contract
lengths, subscription cycles), making it a great example to reuse in interviews.

In [ ]:
seasons_int = df['tv_seasons'].dropna().astype('int')
season_counts = seasons_int.value_counts().sort_index()
print(season_counts)

fig, ax = plt.subplots()
sns.countplot(x=seasons_int, color='#E50914', ax=ax)
ax.set_title('Number of seasons across TV Shows', pad=12)
ax.set_xlabel('Seasons')
ax.set_ylabel('Number of shows')
plt.savefig(FIG_DIR / 'tv_seasons.png', bbox_inches='tight')
plt.show()

**What this cell does**

Answers a bonus question: how is the catalog distributed by release decade? A long tail of older
licensed titles plus a strong 2010s/2020s block shows Netflix combines classic catalog content
with a push of recent originals.

**Pandas functions used**

- `Series.value_counts().sort_index()` — decade counts in chronological order.
- `Series.index.astype(str)` — converts the decade labels for clean axis ticks.
- `sns.barplot` — compares bucket sizes visually.

**How to answer this in an interview**

Tie it to a business story: the 2010s/2020s bulge reflects recent originals and viewers preferring
current content, while the 1980s/1990s tail is the licensed library. Interpreting a chart in
business terms is exactly what an interviewer screens for.

In [ ]:
decade_counts = df['release_decade'].value_counts().sort_index()
print(decade_counts)

fig, ax = plt.subplots()
sns.barplot(x=decade_counts.index.astype(str), y=decade_counts.values, color='#E50914', ax=ax)
ax.set_title('Catalog size by release decade', pad=12)
ax.set_xlabel('Release decade')
ax.set_ylabel('Number of titles')
plt.savefig(FIG_DIR / 'release_decades.png', bbox_inches='tight')
plt.show()

## 7. Key Insights

**What this cell does**

Aggregates every headline number computed above (shares, top country, top genre, top rating,
median duration, peak year) into one tidy summary table. In a real engagement this table would
become the executive one-pager.

**Pandas functions used**

- `pd.DataFrame({'Metric': [...], 'Value': [...]})` — a dict of parallel lists becomes a table.
- f-strings inside the lists — combine numbers with units and labels at report time.
- `Series.index[0]` / `Series.iloc[0]` — the label and the value of the top category.
- `Series.idxmax()` — the index (label) of the maximum value, e.g. the peak addition year.

**How to answer this in an interview**

'How do you close a data analysis?' — reduce the whole notebook to a single Metric / Value table,
then tell three business stories from it. Interviewers test whether you can synthesise, not just
run code, so always finish an analysis by summarising findings.

In [ ]:
peak_year = yearly.groupby('year_added')['count'].sum().idxmax()

insights = pd.DataFrame({
    'Metric': [
        'Total titles in catalog',
        'Share of Movies',
        'Share of TV Shows',
        'Top producing country',
        'Most common genre',
        'Most common rating',
        'Median movie duration (min)',
        'Most common season count',
        'Peak year for additions',
    ],
    'Value': [
        f'{len(df):,}',
        f"{type_share.get('Movie', 0):.1f}%",
        f"{type_share.get('TV Show', 0):.1f}%",
        f'{country_counts.index[0]} ({country_counts.iloc[0]:,})',
        f'{genre_counts.index[0]} ({genre_counts.iloc[0]:,})',
        f'{rating_counts.index[0]} ({rating_counts.iloc[0]:,})',
        f'{movie_dur.median():.0f} min',
        f'{season_counts.idxmax()}',
        f'{peak_year}',
    ],
})
insights

### What the data tells us

- **Movies dominate** — roughly 70% of the catalog is films; TV shows make up the rest and have
  been growing faster in recent years.
- **Strong mid-2010s growth** — titles added per year accelerated sharply from 2015 onward,
  matching Netflix's global expansion push, and peaked around 2019-2020.
- **US-led production** — the United States is the largest source of content, with India a clear
  second, reflecting Netflix's two biggest markets.
- **Dramas and Comedies rule** — the most common genres are Dramas, Comedies and Documentaries,
  which anchor the catalog even though individual 'hits' are often action films.
- **Teen/adult skew** — TV-MA and TV-14 are the most frequent ratings, so the catalog targets
  teens and adults more than young children.
- **Standard runtimes** — the median movie is around 98 minutes; the median TV show has a single
  season, with multi-season shows a minority.

> Note: if the synthetic fallback was used, every figure above is a *demo* of the technique — the
> real dataset produces the true numbers. Re-run with `data/netflix_titles.csv` to verify.

## 8. Conclusion

This notebook walked the full analytics workflow — load, explore, clean, analyse, and report —
against the Netflix catalog and produced six business takeaways:

1. **Content mix: ~70/30** movies vs TV shows.
2. **Rapid 2015+ growth** in titles added per year.
3. **US + India lead** production, with a wide long-tail of countries.
4. **Dramas/Comedies dominate** genre frequency.
5. **Ratings skew teen/adult** (TV-MA, TV-14).
6. **Standard runtimes**: ~98-minute median movie, single-season-led shows.

**What made this analysis possible (the key Pandas skills)**

- `read_csv` / `shape` / `head` / `info` / `describe` — fast dataset orientation.
- `isnull().sum()` / `duplicated()` / `fillna()` — data-quality management.
- `to_datetime` + the `.dt` accessor — turning text dates into an analyzable time series.
- `str.split` + `explode` — taming comma-separated multi-value fields.
- `groupby().size()` and `value_counts(normalize=True)` — the bread and butter of categorical EDA.

**Limitations and honest next steps**

- The snapshot is from ~2021, so the catalog is far larger today and some conclusions age quickly.
- `date_added`, `country`, and `rating` contain nulls / multi-values; we handled them explicitly,
  but the assumptions (e.g. filling with `'Unknown'`) shape the results.
- Suggested next steps: (1) NLP topic modelling on `description`; (2) joining a revenue or watch
  counts dataset for value analysis; (3) moving these charts into a BI dashboard.

**What this cell does**

Prints the exact library versions used so anyone re-running this notebook can reproduce your
results (or understand version drift). It also prints the final shape of the cleaned table as a
closing sanity check.

**Pandas functions used**

- `pd.__version__` — the version attribute of the pandas module (same pattern for numpy,
  matplotlib, and seaborn).
- `DataFrame.shape` — the final row/column count after cleaning.

**How to answer this in an interview**

'How do you make an analysis reproducible?' — pin library versions (`requirements.txt` or
`pd.__version__`), fix the random seed for any sampling, and keep raw vs derived data separate.
Reproducibility answers are a strong, low-effort differentiator.

In [ ]:
print(f'pandas {pd.__version__}')
print(f'numpy {np.__version__}')
print(f'matplotlib {matplotlib.__version__}')
print(f'seaborn {sns.__version__}')
print(f'
Cleaned data: {df.shape[0]:,} rows x {df.shape[1]:,} columns')